# **Fourth Homework Assignment** — Lenia with MPI

In [1]:
import numpy as np
import pandas as pd
import re

# Used for setting display settings to increase readability.
from IPython.core.display import display_html

In [2]:
# Custom formater.
def fromat_nan(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        return f"{round(val, 3)}"
    return f"{val}"

# Helper function that displays each dataframe in its own column.
def display_list(dfs, index=True, axis=-1, minmax="min"):
    # Convert each split to HTML with proper styling for side-by-side display.
    html_str = "<table><tr>" 
    for df in dfs:
        # Dataframe styling options.
        styled_df = df.style.format(fromat_nan)
        if axis >= 0 and minmax == "min":
            styled_df = styled_df.highlight_min(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_max(axis=axis, color='darkred')
        if axis >= 0 and minmax == "max":
            styled_df = styled_df.highlight_max(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_min(axis=axis, color='darkred')
        styled_html = styled_df.to_html(index=index)

        # Enables newlines and bold titles.
        name_html = df.attrs['name'].replace('\n', '<br>')
        name_html = f"<div style='text-align: left; font-weight: bold;'>{name_html}</div>"
        html_str += f"<td style='vertical-align: top; padding: 5px;'>{name_html + styled_html}</td>"
    html_str += "</tr></table>"
    display_html(html_str, raw=True)
    
# Helper function to display one data frame as multiple columns.
def display(df, cols, index=True):
    # Split the dataframe into columns.
    ratio = int(np.ceil(len(df) / cols))
    df_split = [df.iloc[i * ratio: (i + 1) * ratio] for i in range(cols)]
    display_list(df_split, index)


In [3]:
# Calculates means based on axis and returns the dataframe.
def get_means(dfs, axis):
    lst = []
    for df in dfs:
        mean = df.mean(axis).to_frame(name='Mean')
        mean.attrs['name'] = df.attrs['name']
        lst.append(mean)
    return lst

## **Time Comparisons**

In [4]:
import glob
from pathlib import Path

# CSV columns produced by benchmarks/bench.sh: Run,Size,Method,Procs,Nodes,Halo,Time
_csv = pd.concat([pd.read_csv(f) for f in glob.glob("results_*.csv")], ignore_index=True)
_csv.columns = [c.strip() for c in _csv.columns]
_csv["Time"] = pd.to_numeric(_csv["Time"], errors="coerce")
for c in ("Size", "Procs", "Nodes", "Halo"):
    _csv[c] = pd.to_numeric(_csv[c], errors="coerce").astype("Int64")
_csv = _csv.dropna(subset=["Time"])

# Mean across runs for every (Size, Method, Procs, Nodes, Halo) cell.
_means = _csv.groupby(["Size", "Method", "Procs", "Nodes", "Halo"])["Time"].mean()

sizes = [128, 512, 1024, 2048, 4096]
procs = [1, 2, 4, 16, 32]

### Average execution times

For each MPI variant (`row`, `block`) the table shows the **average execution time** (in seconds) across all repeated runs for every combination on **one node** with halo width K = 1. The first column **`seq`** is the optimized sequential reference in `lenia_seq.c` and is what serves as t<sub>s</sub> in the speed-up calculation below. The remaining columns are the MPI variant launched with P = 1, 2, 4, 16, 32 ranks. Comparing `seq` against the P = 1 column shows the cost of MPI bookkeeping at zero parallelism.

In [10]:
def _t(method, size, p, nodes=1, halo=1):
    key = (size, method, p, nodes, halo)
    return _means[key] if key in _means.index else np.nan

# Average execution time per (Size, Procs) on a single node, halo=1.
# First column "seq" is the pure sequential baseline (lenia_seq.c, no MPI);
# the remaining columns are the MPI variant run with P = 1, 2, 4, 16, 32 ranks.
def _time_table(method):
    cols = ["seq"] + procs
    df = pd.DataFrame(index=sizes, columns=cols, dtype=float)
    df.index.name = "Size"
    for s in sizes:
        df.loc[s, "seq"] = _t("seq", s, 1)
        for p in procs:
            df.loc[s, p] = _t(method, s, p)
    df.attrs["name"] = f"Average execution time (s), method = {method}"
    return df

print("Maximal values per row are displayed in red, while minimal are displayed in green.")
display_list([_time_table("row"), _time_table("block")], axis=1)

Maximal values per row are displayed in red, while minimal are displayed in green.


,seq,1,2,4,16,32
Size,,,,,,
128,1.179,1.143,0.595,0.362,0.136,
512,19.486,19.307,9.216,5.558,1.242,0.712
1024,76.951,75.051,37.431,22.239,7.244,2.931
2048,312.088,294.965,147.918,73.451,19.309,10.291
4096,1557.646,1587.704,790.321,392.969,96.215,48.492
,seq,1,2,4,16,32
Size,,,,,,
128,1.179,1.012,0.509,0.263,0.07,0.039
512,19.486,16.128,8.072,4.145,1.032,0.527


In [11]:
# Speed-up S = t_s / t_p(P) where t_s is the sequential baseline.
def _speedup_table(method):
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        ts = _t("seq", s, 1)
        for p in procs:
            tp = _t(method, s, p)
            df.loc[s, p] = (ts / tp) if (tp and tp > 0) else np.nan
    df.attrs["name"] = f"Speed-up t_s / t_p, method = {method}"
    return df

print("Maximal values per row are displayed in green, while minimal are displayed in red.")
display_list([_speedup_table("row"), _speedup_table("block")], axis=1, minmax="max")

Maximal values per row are displayed in green, while minimal are displayed in red.


Procs,1,2,4,16,32
Size,,,,,
128,1.031,1.981,3.261,8.683,
512,1.009,2.114,3.506,15.689,27.368
1024,1.025,2.056,3.46,10.623,26.252
2048,1.058,2.11,4.249,16.163,30.327
4096,0.981,1.971,3.964,16.189,32.122
Procs,1,2,4,16,32
Size,,,,,
128,1.165,2.316,4.487,16.943,30.236
512,1.208,2.414,4.701,18.889,36.989


### Single-node vs two-node runs

In [16]:
# 1-node vs 2-nodes for the same total process count (bonus).
# Same Size x Procs layout as the avg-time tables above, but split by node count
# so 1-node and 2-node times can be compared cell-by-cell. We only queued 2-node
# runs for sizes {1024, 4096} and procs {16, 32}; the other 2-node cells stay blank.
def _nodes_table(method, nodes):
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        for p in procs:
            df.loc[s, p] = _t(method, s, p, nodes=nodes)
    suffix = f"{nodes} node" + ("s" if nodes > 1 else "")
    df.attrs["name"] = f"Average execution time (s), method = {method}, {suffix}"
    return df

print("Method = row")
display_list([_nodes_table("row", 1), _nodes_table("row", 2)])

print("Method = block")
display_list([_nodes_table("block", 1), _nodes_table("block", 2)])

Method = row


Procs,1,2,4,16,32
Size,,,,,
128,1.143,0.595,0.362,0.136,
512,19.307,9.216,5.558,1.242,0.712
1024,75.051,37.431,22.239,7.244,2.931
2048,294.965,147.918,73.451,19.309,10.291
4096,1587.704,790.321,392.969,96.215,48.492
Procs,1,2,4,16,32
Size,,,,,
128,,,,,
512,,,,,


Method = block


Procs,1,2,4,16,32
Size,,,,,
128,1.012,0.509,0.263,0.07,0.039
512,16.128,8.072,4.145,1.032,0.527
1024,64.487,32.263,16.404,4.112,2.045
2048,258.114,129.098,64.593,16.378,8.112
4096,1032.912,516.593,258.57,66.632,33.287
Procs,1,2,4,16,32
Size,,,,,
128,,,,,
512,,,,,


### Wide-halo communication-overhead bonus

The `lenia_row_wide` variant exchanges a halo of width K · R every K iterations instead of width R every iteration. It trades **K× more boundary computation per exchange** for **K× fewer halo exchanges** over the simulation. The table shows average execution time at a fixed point for K ∈ {1, 2, 4, 8} so we can see whether reducing exchange frequency wins on this cluster, meaning whether the per-message latency was a meaningful share of the original runtime.

In [17]:
# Wide-halo K sweep (bonus). Halo width = K * R, exchange every K iterations.
_wide = _csv[_csv["Method"] == "row_wide"].copy()
if not _wide.empty:
    df_wide = _wide.groupby(["Size", "Procs", "Halo"])["Time"].mean().unstack("Halo")
    df_wide.attrs["name"] = "Wide-halo (lenia_row_wide): mean time per K, indexed by (Size, Procs)"
    display_list([df_wide], axis=1)
else:
    print("No row_wide results found yet.")

,Halo,1,2,4,8
Size,Procs,,,,
2048,16,19.493,21.124,24.964,32.678
4096,32,49.976,54.861,68.485,108.591
